In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

class WinePredictor:
    def __init__(self):
        # Load and inspect dataset
        self.df = pd.read_csv('C:\\Users\\Varsha\\Downloads\\winecolor.csv')
        print("Available columns in dataset:", self.df.columns)

        # Check if required columns are present
        required_columns = ['type', 'color_intensity', 'quality']
        for col in required_columns:
            if col not in self.df.columns:
                raise KeyError(f"Expected column '{col}' not found in dataset.")

        # Label encoding for wine type
        self.le = LabelEncoder()
        self.df['type'] = self.le.fit_transform(self.df['type'])

        # Define age category based on color intensity and type
        self.df['age_category'] = self.df.apply(self.age_based_on_color_intensity, axis=1)

        # Define quality category (assuming quality is on a scale, e.g., 0-10)
        self.df['quality_category'] = self.df['quality'].apply(
            lambda x: 'Good' if x >= 6 else 'Bad'
        )

        # Convert categories to numerical values
        self.age_le = LabelEncoder()
        self.quality_le = LabelEncoder()
        self.df['age_category_encoded'] = self.age_le.fit_transform(self.df['age_category'])
        self.df['quality_category_encoded'] = self.quality_le.fit_transform(self.df['quality_category'])

        # Select features for prediction (excluding quality-related columns)
        self.feature_columns = [col for col in self.df.columns 
                              if col not in ['age_category', 'age_category_encoded', 
                                           'quality', 'quality_category', 'quality_category_encoded']]
        
        print("\nFeatures used for prediction:", self.feature_columns)
        
        self.X = self.df[self.feature_columns]
        self.y_age = self.df['age_category_encoded']
        self.y_quality = self.df['quality_category_encoded']

        # Display distributions
        print("\nAge Distribution in Dataset:")
        print(self.df['age_category'].value_counts())
        print("\nQuality Distribution in Dataset:")
        print(self.df['quality_category'].value_counts())

        # Split data
        self.X_train, self.X_test, self.y_age_train, self.y_age_test, \
        self.y_quality_train, self.y_quality_test = train_test_split(
            self.X, self.y_age, self.y_quality, 
            test_size=0.2, random_state=42
        )

        # Initialize models for age prediction
        self.rf_age_model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.xgb_age_model = XGBClassifier(
            n_estimators=100,
            random_state=42,
            objective='multi:softmax',
            num_class=len(self.df['age_category'].unique())
        )

        # Initialize models for quality prediction
        self.rf_quality_model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.xgb_quality_model = XGBClassifier(
            n_estimators=100,
            random_state=42,
            objective='multi:softmax',
            num_class=len(self.df['quality_category'].unique())
        )

        print("\nTraining models...")
        # Train age prediction models
        self.rf_age_model.fit(self.X_train, self.y_age_train)
        self.xgb_age_model.fit(self.X_train, self.y_age_train)
        
        # Train quality prediction models
        self.rf_quality_model.fit(self.X_train, self.y_quality_train)
        self.xgb_quality_model.fit(self.X_train, self.y_quality_train)

        # Evaluate models
        print("\nAge Prediction Performance:")
        self._evaluate_models(
            self.rf_age_model, self.xgb_age_model,
            self.X_test, self.y_age_test,
            self.age_le.classes_,
            "Age"
        )

        print("\nQuality Prediction Performance:")
        self._evaluate_models(
            self.rf_quality_model, self.xgb_quality_model,
            self.X_test, self.y_quality_test,
            self.quality_le.classes_,
            "Quality"
        )

        # Print feature importance for quality prediction
        self._print_feature_importance()

    def _evaluate_models(self, rf_model, xgb_model, X_test, y_test, class_names, prediction_type):
        """Helper method to evaluate models"""
        rf_pred = rf_model.predict(X_test)
        xgb_pred = xgb_model.predict(X_test)

        print(f"\nRandom Forest {prediction_type} Performance:")
        print(classification_report(y_test, rf_pred, target_names=class_names))
        
        print(f"\nXGBoost {prediction_type} Performance:")
        print(classification_report(y_test, xgb_pred, target_names=class_names))

    def _print_feature_importance(self):
        """Print feature importance for quality prediction"""
        # Get feature importance from Random Forest model
        importance = self.rf_quality_model.feature_importances_
        features = self.feature_columns
        
        # Sort features by importance
        feature_imp = sorted(zip(importance, features), reverse=True)
        
        print("\nFeature Importance for Quality Prediction (Random Forest):")
        for imp, feature in feature_imp:
            print(f"{feature}: {imp:.4f}")

    @staticmethod
    def age_based_on_color_intensity(row):
        """Determine wine age category based on color intensity and type"""
        if row['type'] == 0:  # red wine
            return 'Fresh' if row['color_intensity'] > 5 else 'Old'
        else:  # white wine
            return 'Old' if row['color_intensity'] > 5 else 'Fresh'

    def predict(self, wine_type, feature_inputs):
        """
        Predict both age and quality categories using ensemble models
        
        Parameters:
        wine_type (str): Type of wine ('red' or 'white')
        feature_inputs (dict): Dictionary containing feature values (excluding quality)
        
        Returns:
        dict: Predictions for both age and quality
        """
        # Convert wine type to numerical
        wine_type_num = self.le.transform([wine_type])[0]
        
        # Create input data array matching the feature columns order
        input_data = []
        for feature in self.feature_columns:
            if feature == 'type':
                input_data.append(wine_type_num)
            else:
                input_data.append(feature_inputs.get(feature, 0))
        
        # Reshape to meet model input shape
        input_data = np.array(input_data).reshape(1, -1)
        
        # Get age predictions
        rf_age_pred = self.rf_age_model.predict(input_data)[0]
        xgb_age_pred = self.xgb_age_model.predict(input_data)[0]
        
        # Get quality predictions
        rf_quality_pred = self.rf_quality_model.predict(input_data)[0]
        xgb_quality_pred = self.xgb_quality_model.predict(input_data)[0]

        # Decode predictions
        rf_age = self.age_le.inverse_transform([rf_age_pred])[0]
        xgb_age = self.age_le.inverse_transform([xgb_age_pred])[0]
        rf_quality = self.quality_le.inverse_transform([rf_quality_pred])[0]
        xgb_quality = self.quality_le.inverse_transform([xgb_quality_pred])[0]

        # Ensemble predictions (simple majority voting)
        final_age = rf_age if rf_age == xgb_age else rf_age
        final_quality = rf_quality if rf_quality == xgb_quality else rf_quality

        return {
            'age_prediction': {
                'random_forest': rf_age,
                'xgboost': xgb_age,
                'ensemble': final_age
            },
            'quality_prediction': {
                'random_forest': rf_quality,
                'xgboost': xgb_quality,
                'ensemble': final_quality
            }
        }

# Initialize and test
if __name__ == "__main__":
    try:
        predictor = WinePredictor()
    except KeyError as e:
        print(e)
    else:
        print("\nPlease enter wine characteristics:")
        wine_type = input("Wine type (red/white): ")
        
        # Collect all features in a dictionary (excluding quality)
        feature_inputs = {}
        for feature in predictor.feature_columns:
            if feature != 'type':  # Skip type as it's handled separately
                feature_inputs[feature] = float(input(f"{feature.replace('_', ' ').title()}: "))

        # Get predictions
        predictions = predictor.predict(wine_type, feature_inputs)
        
        print("\nPrediction Results:")
        print("\nAge Prediction:")
        print(f"Random Forest: {predictions['age_prediction']['random_forest']}")
        print(f"XGBoost: {predictions['age_prediction']['xgboost']}")
        print(f"Final Ensemble: {predictions['age_prediction']['ensemble']}")
        
        print("\nQuality Prediction:")
        print(f"Random Forest: {predictions['quality_prediction']['random_forest']}")
        print(f"XGBoost: {predictions['quality_prediction']['xgboost']}")
        print(f"Final Ensemble: {predictions['quality_prediction']['ensemble']}") 




Available columns in dataset: Index(['type', 'fixed acidity', 'volatile acidity', 'citric acid',
       'residual sugar', 'chlorides', 'free sulfur dioxide',
       'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol',
       'quality', 'color_intensity'],
      dtype='object')

Features used for prediction: ['type', 'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'color_intensity']

Age Distribution in Dataset:
age_category
Old      3315
Fresh    3182
Name: count, dtype: int64

Quality Distribution in Dataset:
quality_category
Good    4113
Bad     2384
Name: count, dtype: int64

Training models...

Age Prediction Performance:

Random Forest Age Performance:
              precision    recall  f1-score   support

       Fresh       1.00      1.00      1.00       655
         Old       1.00      1.00      1.00       645

    accuracy                       

ValueError: could not convert string to float: ''